# CleanRL's Huggingface Integration Demo



[<img src="https://img.shields.io/badge/license-MIT-blue">](https://github.com/vwxyzjn/cleanrl)
[![tests](https://github.com/vwxyzjn/cleanrl/actions/workflows/tests.yaml/badge.svg)](https://github.com/vwxyzjn/cleanrl/actions/workflows/tests.yaml)
[![docs](https://img.shields.io/github/deployments/vwxyzjn/cleanrl/Production?label=docs&logo=vercel)](https://docs.cleanrl.dev/)
[<img src="https://img.shields.io/discord/767863440248143916?label=discord">](https://discord.gg/D6RCjA6sVT)
[<img src="https://img.shields.io/youtube/channel/views/UCDdC6BIFRI0jvcwuhi3aI6w?style=social">](https://www.youtube.com/channel/UCDdC6BIFRI0jvcwuhi3aI6w/videos)
[![Code style: black](https://img.shields.io/badge/code%20style-black-000000.svg)](https://github.com/psf/black)
[![Imports: isort](https://img.shields.io/badge/%20imports-isort-%231674b1?style=flat&labelColor=ef8336)](https://pycqa.github.io/isort/)
[<img src="https://img.shields.io/badge/%F0%9F%A4%97%20Models-Huggingface-F8D521">](https://huggingface.co/cleanrl)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vwxyzjn/cleanrl/blob/master/docs/get-started/CleanRL_Huggingface_Integration_Demo.ipynb)


CleanRL is a Deep Reinforcement Learning library that provides high-quality single-file implementation with research-friendly features. It now has has 🧪 experimental support for saving and loading models from 🤗 HuggingFace's [Model Hub](https://huggingface.co/models). This notebook is a preliminary demo.


* 💾 [GitHub Repo](https://github.com/vwxyzjn/cleanrl)
* 📜 [Documentation](https://docs.cleanrl.dev/)
* 🤗 [HuggingFace Model Hub](https://huggingface.co/cleanrl)
* 🔗 [Open RL Benchmark reports](https://wandb.ai/openrlbenchmark/openrlbenchmark/reportlist)



## Get Started

CleanRL can be installed via `pip`. Let's say we are interested in pulling the model for [`dqn_atari_jax.py`](https://github.com/vwxyzjn/cleanrl/blob/master/cleanrl/dqn_atari_jax.py), we can install the algorithm-variant-specific dependencies as follows:

In [ ]:
!git clone https://github.com/ana-lys/cleanrlAIS2603.git
%cd cleanrlAIS2603
!pip install -e .

In [5]:
import wandb
import os
from google.colab import userdata

# --- SET YOUR CREDENTIALS HERE ---
# 1. Paste your API Key here
MY_WANDB_API_KEY = "your_long_api_key_here"

# 2. Enter your W&B Username (Entity) here
USER_ENTITY = "your_user_name here"

# Login logic
if MY_WANDB_API_KEY == "your_long_api_key_here":
    print("❌ Error: You forgot to paste your API Key in the code block!")
else:
    os.environ["WANDB_API_KEY"] = MY_WANDB_API_KEY
    wandb.login()
    print(f"✅ Logged in as: {USER_ENTITY}")


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: trandzuo3052 (trandzuo3052-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Logged in as: trandzuo3052


3. The PPO Algorithm Classes
This block defines your Args, Agent, and helper functions.


In [19]:
import random
import time
from dataclasses import dataclass
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import tyro
from torch.distributions.categorical import Categorical
from torch.utils.tensorboard import SummaryWriter

@dataclass
class Args:
    tag: str = "v2"
    exp_name: str = "ppo_colab"
    seed: int = 99
    torch_deterministic: bool = True
    cuda: bool = True
    track: bool = True
    wandb_project_name: str = "cleanRL"
    wandb_entity: str = None # Replace with your W&B username if needed

    # Algorithm specific
    env_id: str = "CartPole-v1"
    total_timesteps: int = 1500000
    learning_rate: float = 5.0e-4
    num_envs: int = 512 # Reduced for standard Colab stability
    num_steps: int = 256
    anneal_lr: bool = False
    gamma: float = 0.99
    gae_lambda: float = 0.95
    num_minibatches: int = 64
    update_epochs: int = 4
    norm_adv: bool = True
    clip_coef: float = 0.2
    clip_vloss: bool = False
    ent_coef: float = 0.01
    vf_coef: float = 0.5
    max_grad_norm: float = 0.5
    target_kl: float = None

    # Runtime computed
    batch_size: int = 0
    minibatch_size: int = 0
    num_iterations: int = 0

def make_env(env_id, idx):
    def thunk():
        env = gym.make(env_id)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        return env
    return thunk

def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class Agent(nn.Module):
    def __init__(self, envs):
        super().__init__()
        obs_shape = np.array(envs.single_observation_space.shape).prod()
        self.critic = nn.Sequential(
            layer_init(nn.Linear(obs_shape, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 1), std=1.0),
        )
        self.actor = nn.Sequential(
            layer_init(nn.Linear(obs_shape, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, envs.single_action_space.n), std=1.0),
        )

    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(self, x, action=None):
        logits = self.actor(x)
        probs = Categorical(logits=logits)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(x)

4. Training Loop Execution
This cell runs the actual training. You can modify hyperparameters here before running.

In [20]:
# 1. Initialize Args (using tyro to compute batch sizes)
args = tyro.cli(Args, args=[])
args.batch_size = int(args.num_envs * args.num_steps)
args.minibatch_size = int(args.batch_size // args.num_minibatches)
args.num_iterations = args.total_timesteps // args.batch_size
run_name = f"{args.env_id}__{args.exp_name}__{args.seed}__{int(time.time())}"

# 2. Setup Logging (W&B and Tensorboard)
if args.track:
    import wandb
    wandb.init(
        project=args.wandb_project_name,
        entity=args.wandb_entity,
        sync_tensorboard=True,
        config=vars(args),
        name=run_name,
        monitor_gym=False,
        save_code=True,
    )

writer = SummaryWriter(f"runs/{run_name}")
writer.add_text(
    "hyperparameters",
    "|param|value|\n|-|-|\n%s"
    % ("\n".join([f"|{key}|{value}|" for key, value in vars(args).items()])),
)

# 3. Seeding & Hardware Setup
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.backends.cudnn.deterministic = args.torch_deterministic
device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")

# 4. Env & Agent Initialization
envs = gym.vector.SyncVectorEnv(
    [make_env(args.env_id, i) for i in range(args.num_envs)],
    autoreset_mode=gym.vector.AutoresetMode.SAME_STEP,
)
agent = Agent(envs).to(device)
optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate, eps=1e-5)

# 5. Storage Setup
obs = torch.zeros((args.num_steps, args.num_envs) + envs.single_observation_space.shape).to(device)
actions = torch.zeros((args.num_steps, args.num_envs) + envs.single_action_space.shape).to(device)
logprobs = torch.zeros((args.num_steps, args.num_envs)).to(device)
rewards = torch.zeros((args.num_steps, args.num_envs)).to(device)
dones = torch.zeros((args.num_steps, args.num_envs)).to(device)
values = torch.zeros((args.num_steps, args.num_envs)).to(device)

# 6. Start the Game
global_step = 0
start_time = time.time()
next_obs, _ = envs.reset(seed=args.seed)
next_obs = torch.Tensor(next_obs).to(device)
next_done = torch.zeros(args.num_envs).to(device)

print(f"Starting training on {device}...")

# --- CORE TRAINING LOOP ---
for iteration in range(1, args.num_iterations + 1):
    # Annealing the rate if toggled
    if args.anneal_lr:
        frac = 1.0 - (iteration - 1.0) / args.num_iterations
        lrnow = frac * args.learning_rate
        optimizer.param_groups[0]["lr"] = lrnow

    for step in range(0, args.num_steps):
        global_step += args.num_envs
        obs[step] = next_obs
        dones[step] = next_done

        # Action logic
        with torch.no_grad():
            action, logprob, _, value = agent.get_action_and_value(next_obs)
            values[step] = value.flatten()
        actions[step] = action
        logprobs[step] = logprob

        # Step the environment
        next_obs, reward, terminations, truncations, infos = envs.step(action.cpu().numpy())
        next_done = np.logical_or(terminations, truncations)
        rewards[step] = torch.tensor(reward).to(device).view(-1)
        next_obs, next_done = torch.Tensor(next_obs).to(device), torch.Tensor(next_done).to(device)

        # Logging episodic returns
        if "final_info" in infos:
            final_info = infos["final_info"]
            if isinstance(final_info, dict) and "episode" in final_info:
                ep_dict = final_info["episode"]
                mask = ep_dict["_episode"] if "_episode" in ep_dict else (ep_dict["r"] != 0)

                finished_r = ep_dict["r"][mask]
                finished_l = ep_dict["l"][mask]
                if len(finished_r) > 0:
                    avg_return = np.mean(finished_r)
                    avg_length = np.mean(finished_l)
                    writer.add_scalar("charts/avg_episodic_return", avg_return, global_step)
                    writer.add_scalar("charts/avg_episodic_length", avg_length, global_step)

    # 7. Generalized Advantage Estimation (GAE)
    with torch.no_grad():
        next_value = agent.get_value(next_obs).reshape(1, -1)
        advantages = torch.zeros_like(rewards).to(device)
        lastgaelam = 0
        for t in reversed(range(args.num_steps)):
            if t == args.num_steps - 1:
                nextnonterminal = 1.0 - next_done
                nextvalues = next_value
            else:
                nextnonterminal = 1.0 - dones[t + 1]
                nextvalues = values[t + 1]
            delta = rewards[t] + args.gamma * nextvalues * nextnonterminal - values[t]
            advantages[t] = lastgaelam = (
                delta + args.gamma * args.gae_lambda * nextnonterminal * lastgaelam
            )
        returns = advantages + values

    # 8. Optimization Logic
    b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
    b_logprobs = logprobs.reshape(-1)
    b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
    b_advantages = advantages.reshape(-1)
    b_returns = returns.reshape(-1)
    b_values = values.reshape(-1)

    b_inds = np.arange(args.batch_size)
    clipfracs = []
    for epoch in range(args.update_epochs):
        np.random.shuffle(b_inds)
        for start in range(0, args.batch_size, args.minibatch_size):
            end = start + args.minibatch_size
            mb_inds = b_inds[start:end]

            _, newlogprob, entropy, newvalue = agent.get_action_and_value(
                b_obs[mb_inds], b_actions.long()[mb_inds]
            )
            logratio = newlogprob - b_logprobs[mb_inds]
            ratio = logratio.exp()

            with torch.no_grad():
                old_approx_kl = (-logratio).mean()
                approx_kl = ((ratio - 1) - logratio).mean()
                clipfracs += [((ratio - 1.0).abs() > args.clip_coef).float().mean().item()]

            mb_advantages = b_advantages[mb_inds]
            if args.norm_adv:
                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

            # Policy loss
            pg_loss1 = -mb_advantages * ratio
            pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - args.clip_coef, 1 + args.clip_coef)
            pg_loss = torch.max(pg_loss1, pg_loss2).mean()

            # Value loss
            newvalue = newvalue.view(-1)
            if args.clip_vloss:
                v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
                v_clipped = b_values[mb_inds] + torch.clamp(
                    newvalue - b_values[mb_inds], -args.clip_coef, args.clip_coef
                )
                v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
                v_loss = 0.5 * torch.max(v_loss_unclipped, v_loss_clipped).mean()
            else:
                v_loss = 0.5 * ((newvalue - b_returns[mb_inds]) ** 2).mean()

            entropy_loss = entropy.mean()
            loss = pg_loss - args.ent_coef * entropy_loss + v_loss * args.vf_coef

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), args.max_grad_norm)
            optimizer.step()

        if args.target_kl is not None and approx_kl > args.target_kl:
            break

   # 9. Metrics Post-Iteration (Restored All Metrics)
    y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
    var_y = np.var(y_true)
    explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y

    # Metrics logged to SummaryWriter (Syncs to W&B)
    writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
    writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
    writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
    writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
    writer.add_scalar("losses/old_approx_kl", old_approx_kl.item(), global_step)
    writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
    writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step)
    writer.add_scalar("losses/explained_variance", explained_var, global_step)

    # Performance logging
    sps = int(global_step / (time.time() - start_time))
    writer.add_scalar("charts/SPS", sps, global_step)
    writer.add_scalar("charts/wall_time", int(time.time() - start_time), global_step)

    # Console Output
    print(f"SPS: {sps} | iteration: {iteration} | global_step: {global_step}")

# 10. Shutdown
envs.close()
writer.close()
if args.track:
    wandb.finish()

/usr/local/lib/python3.12/dist-packages/tyro/_parsers.py:353: UserWarning: The field `wandb-entity` is annotated with type `<class 'str'>`, but the default value `None` has type `<class 'NoneType'>`. We'll try to handle this gracefully, but it may cause unexpected behavior.
  warnings.warn(message)
/usr/local/lib/python3.12/dist-packages/tyro/_parsers.py:353: UserWarning: The field `target-kl` is annotated with type `<class 'float'>`, but the default value `None` has type `<class 'NoneType'>`. We'll try to handle this gracefully, but it may cause unexpected behavior.
  warnings.warn(message)


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


Starting training on cuda...
SPS: 32774 | iteration: 1 | global_step: 131072
SPS: 28526 | iteration: 2 | global_step: 262144
SPS: 29914 | iteration: 3 | global_step: 393216
SPS: 30856 | iteration: 4 | global_step: 524288
SPS: 29402 | iteration: 5 | global_step: 655360
SPS: 30184 | iteration: 6 | global_step: 786432
SPS: 30781 | iteration: 7 | global_step: 917504
SPS: 30156 | iteration: 8 | global_step: 1048576
SPS: 30610 | iteration: 9 | global_step: 1179648
SPS: 30963 | iteration: 10 | global_step: 1310720
SPS: 30792 | iteration: 11 | global_step: 1441792


charts/SPS,█▁▃▅▂▄▅▄▄▅▅
charts/avg_episodic_length,▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▃▂▂▄▄▄▄▄▄▅▇▇█▇███████████
charts/avg_episodic_return,▁▁▁▁▁▁▂▁▁▁▂▂▂▂▂▄▃▄▄▆▇█▇▇████████████████
charts/learning_rate,▁▁▁▁▁▁▁▁▁▁▁
charts/wall_time,▁▂▃▃▄▅▅▆▇▇█
global_step,▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇▇▇█████
losses/approx_kl,▂▅▇█▅▆▃▂▁▁▂
losses/clipfrac,▃▃██▆▅▃▁▁▁▁
losses/entropy,█▇▅▄▃▂▃▁▁▃▂
losses/explained_variance,▁▃▇▇▇█▇▅▃▂▂
+3,...
